In [1]:
# -*- coding: utf-8 -*-
"""Untitled28.ipynb"""

import pandas as pd, numpy as np, math, os
import scipy.optimize as opt
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment, PatternFill

path="/content/PREÇO MED DIARIO PLD VERTICAL PERIODO 17-04-18 A 03-06-25.xlsx"
df=pd.read_excel(path)

# detect columns
cols=list(df.columns)
date_col=None; price_col=None
for c in cols:
    cl=str(c).strip().lower()
    if date_col is None and "data" in cl:
        date_col=c
    if ("preço" in cl or "preco" in cl or "pld" in cl):
        if price_col is None or "medio" in cl or "médio" in cl:
            price_col=c

if date_col is None:
    for c in cols:
        if np.issubdtype(df[c].dtype, np.datetime64):
            date_col=c; break

if price_col is None:
    for c in cols:
        if np.issubdtype(df[c].dtype, np.number):
            price_col=c; break

df=df.rename(columns={date_col:"data", price_col:"preço_PLD"})
df["data"]=pd.to_datetime(df["data"])
df=df.sort_values("data").reset_index(drop=True)

df["retorno_log"]=np.log(df["preço_PLD"]/df["preço_PLD"].shift(1))
df["volatilidade_realizada_diária"]=df["retorno_log"].abs()

# returns in percent for estimation
r = df["retorno_log"].dropna().values*100.0
mu0=float(np.mean(r))
var0=float(np.var(r))

def gjr_nll_raw(theta, r):

    mu = theta[0]
    omega = math.exp(theta[1])
    alpha = math.exp(theta[2])
    gamma = math.exp(theta[3])
    beta  = 1.0/(1.0+math.exp(-theta[4]))

    eps = r - mu
    h = np.empty_like(eps)

    denom = 1 - alpha - 0.5*gamma - beta
    h[0] = max(var0, 1e-6) if denom <= 1e-6 else max(omega/denom, 1e-6)

    for t in range(1, len(eps)):

        ind = 1.0 if eps[t-1] < 0 else 0.0
        h[t] = omega + (alpha + gamma*ind)*(eps[t-1]**2) + beta*h[t-1]

        if h[t] <= 1e-12:
            h[t] = 1e-12

    ll = -0.5*(np.log(2*math.pi) + np.log(h) + (eps**2)/h)
    nll = -float(np.sum(ll))

    phi = alpha + 0.5*gamma + beta

    if phi >= 0.999:
        nll += 1e6*(phi-0.999)**2

    return nll


# estimação
starts = [
    np.array([mu0, np.log(var0*0.01+1e-6), np.log(0.05), np.log(0.05), math.log(0.90/0.10)]),
    np.array([mu0, np.log(var0*0.05+1e-6), np.log(0.10), np.log(0.10), math.log(0.95/0.05)]),
    np.array([mu0, np.log(var0*0.02+1e-6), np.log(0.03), np.log(0.08), math.log(0.85/0.15)]),
    np.array([mu0, np.log(var0*0.10+1e-6), np.log(0.15), np.log(0.05), math.log(0.80/0.20)]),
]

best=None

for x0 in starts:

    res=opt.minimize(gjr_nll_raw, x0, args=(r,), method="L-BFGS-B", options={"maxiter":4000})

    if best is None or res.fun < best.fun:
        best=res


theta=best.x

mu = theta[0]
omega = math.exp(theta[1])
alpha = math.exp(theta[2])
gamma = math.exp(theta[3])
beta  = 1/(1+math.exp(-theta[4]))

phi = alpha + 0.5*gamma + beta


# filtrar volatilidade
r_full = df["retorno_log"].values*100.0
eps_full = r_full - mu
N=len(df)

h=np.full(N, np.nan)

first = np.where(~np.isnan(eps_full))[0][0]

denom = 1 - alpha - 0.5*gamma - beta
h0 = max(np.nanvar(eps_full), 1e-6) if denom<=1e-6 else max(omega/denom, 1e-6)

h[first]=h0

for t in range(first+1, N):

    if np.isnan(eps_full[t-1]) or np.isnan(h[t-1]):
        h[t]=h[t-1]
        continue

    ind = 1.0 if eps_full[t-1] < 0 else 0.0

    h[t] = omega + (alpha + gamma*ind)*(eps_full[t-1]**2) + beta*h[t-1]

    if h[t] <= 1e-12:
        h[t]=1e-12


# volatilidade diária
df["volatilidade_GJR_diária"]=np.sqrt(h)/100.0


# =============================
# NOVA ESCALA TEMPORAL
# =============================

for n, lab in [(5,"1W"),(21,"1M"),(252,"1Y")]:

    df[f"volatilidade_GJR_{lab}"] = (
        df["volatilidade_GJR_diária"] * math.sqrt(n)
    )

    df[f"volatilidade_GJR_{lab}_anualizada"] = (
        df["volatilidade_GJR_diária"] * math.sqrt(252)
    )


out=df[[
"data",
"preço_PLD",
"retorno_log",
"volatilidade_realizada_diária",
"volatilidade_GJR_diária",
"volatilidade_GJR_1W",
"volatilidade_GJR_1M",
"volatilidade_GJR_1Y",
"volatilidade_GJR_1W_anualizada",
"volatilidade_GJR_1M_anualizada",
"volatilidade_GJR_1Y_anualizada"
]].copy()


# escrever Excel
out_path="/content/PLD_volatilidades_GJR.xlsx"

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:

    out.to_excel(writer, index=False, sheet_name="GJR")

    params=pd.DataFrame({
        "Item":[
            "Modelo (GJR-GARCH(1,1))",
            "Equação",
            "Erro",
            "mu",
            "omega",
            "alpha",
            "gamma",
            "beta",
            "phi=alpha+0.5*gamma+beta"
        ],

        "Valor":[
            "h_t = omega + (alpha + gamma*I_{eps_{t-1}<0})*eps_{t-1}^2 + beta*h_{t-1}",
            "eps_t = r_t - mu ; r_t = 100*retorno_log_t ; sigma_t = sqrt(h_t)/100",
            "Normal",
            mu, omega, alpha, gamma, beta, phi
        ]
    })

    params.to_excel(writer, index=False, sheet_name="Parâmetros")


wb=load_workbook(out_path)

header_fill=PatternFill("solid", fgColor="1F4E79")
header_font=Font(color="FFFFFF", bold=True)

ws=wb["GJR"]
ws.freeze_panes="A2"

for j,cell in enumerate(ws[1], start=1):

    cell.fill=header_fill
    cell.font=header_font
    cell.alignment=Alignment(horizontal="center", vertical="center", wrap_text=True)

    ws.column_dimensions[get_column_letter(j)].width=24 if j>1 else 14


for cell in ws["A"][1:]:
    cell.number_format="dd/mm/yyyy"


ws2=wb["Parâmetros"]

ws2.freeze_panes="A2"

for cell in ws2[1]:

    cell.fill=header_fill
    cell.font=header_font
    cell.alignment=Alignment(horizontal="center", vertical="center", wrap_text=True)

ws2.column_dimensions["A"].width=30
ws2.column_dimensions["B"].width=115

wb.save(out_path)

out_path

'/content/PLD_volatilidades_GJR.xlsx'